In [2]:
"""Adapted from Companion code to https://realpython.com/simulation-with-simpy/

'Simulating Real-World Processes With SimPy'

Python version: 3.7.3
SimPy version: 3.0.11
"""

import simpy
import random
import statistics

In [3]:


class Theater(object):
    def __init__(self, env, num_cashiers, num_servers, num_ushers):
        self.env = env
        ## simpy environment
        
        self.cashier = simpy.Resource(env, num_cashiers)
        # number of cashiers available to sell tickets to the moviegoers.
        
        self.server = simpy.Resource(env, num_servers)
        self.usher = simpy.Resource(env, num_ushers)

    def purchase_ticket(self, moviegoer):
        lam=0.5
        yield self.env.timeout(random.expovariate(lam))

    def check_ticket(self, moviegoer):
        yield self.env.timeout(3 / 60)

    def sell_food(self, moviegoer):
        lam=0.1
        yield self.env.timeout(random.expovariate(lam))



## Agent

In [4]:

def go_to_movies(moviegoer, theater):
    # Moviegoer arrives at the theater
    env=theater.env
    arrival_time = env.now

    
    # with statement helps avoiding bugs and leaks by ensuring that a resource is properly released 
    # when the code using the resource is completely executed.
    with theater.cashier.request() as request:
        # moviegoer generates a request to use a cashier.
        print('%7.4f : person %s waiting for cashier' % (env.now, moviegoer))
        yield request
        # yield request: moviegoer waits for a cashier to become available if all are currently in use.  
        # When a "yield" statement is hit, the program suspends function execution 
        # and returns the yielded value to the caller.
        
        print('%7.4f : person %s got cashier' % (env.now, moviegoer))
        yield env.process(theater.purchase_ticket(moviegoer))
        print('%7.4f : person %s has purchased the ticket ' % (env.now, moviegoer))

    with theater.usher.request() as request:
        print('%7.4f : person %s waiting for usher' % (env.now, moviegoer))
        yield request
        print('%7.4f : person %s got usher' % (env.now, moviegoer))
        yield env.process(theater.check_ticket(moviegoer))
        print('%7.4f : person %s has checked the ticket ' % (env.now, moviegoer))

    if random.choice([True, False]):
        with theater.server.request() as request:
            print('%7.4f : person %s waiting for food service' % (env.now, moviegoer))
            yield request
            print('%7.4f : person %s is being served food' % (env.now, moviegoer))
            yield env.process(theater.sell_food(moviegoer))
            print('%7.4f : person %s got food ' % (env.now, moviegoer))

    # Moviegoer heads into the theater
    delay=env.now - arrival_time
    wait_times.append(delay)
    print('%7.4f : person %s arrived at %7.4f and heads into the theater after  %7.4f minutes' % (env.now, moviegoer,arrival_time,delay))

def run_theater(env, num_cashiers, num_servers, num_ushers):
    theater = Theater(env, num_cashiers, num_servers, num_ushers)
    
    for moviegoer in range(3):
        env.process(go_to_movies( moviegoer, theater))

    lam=2 ## avg. frequency
    while True:
        yield env.timeout(random.expovariate(lam))  # Wait a bit before generating a new person

        moviegoer += 1
        env.process(go_to_movies( moviegoer, theater))






## Analyzing statistics

In [5]:
def get_average_wait_time(wait_times):
    average_wait = statistics.mean(wait_times)
    # Pretty print the results
    minutes, frac_minutes = divmod(average_wait, 1)
    seconds = frac_minutes * 60
    return round(minutes), round(seconds)




In [7]:
wait_times = []
## empty list to hold the wait times 
## Once all the wait times have been appended here, 
## you’ll be able to perform some statistics on this list to determine 
## if you have the proper number of employees or not.


def main():
    # Setup
    random.seed(42)
    num_cashiers=10
    num_servers=1
    num_ushers = 3
    TF= 100 # final time
    
    # Run the simulation
    env = simpy.Environment()
    env.process(run_theater(env, num_cashiers, num_servers, num_ushers))
    env.run(until=TF) 

    # View the results
    mins, secs = get_average_wait_time(wait_times)
    print(
        "Running simulation...",
        f"\nThe average wait time is {mins} minutes and {secs} seconds.",
    )


if __name__ == "__main__":
    main()

 0.0000 : person 0 waiting for cashier
 0.0000 : person 1 waiting for cashier
 0.0000 : person 2 waiting for cashier
 0.0000 : person 0 got cashier
 0.0000 : person 1 got cashier
 0.0000 : person 2 got cashier
 0.0507 : person 0 has purchased the ticket 
 0.0507 : person 0 waiting for usher
 0.0507 : person 0 got usher
 0.1007 : person 0 has checked the ticket 
 0.1007 : person 0 waiting for food service
 0.1007 : person 0 is being served food
 0.5052 : person 2 has purchased the ticket 
 0.5052 : person 2 waiting for usher
 0.5052 : person 2 got usher
 0.5100 : person 3 waiting for cashier
 0.5100 : person 3 got cashier
 0.5552 : person 2 has checked the ticket 
 0.5552 : person 2 arrived at  0.0000 and heads into the theater after   0.5552 minutes
 0.6432 : person 1 has purchased the ticket 
 0.6432 : person 1 waiting for usher
 0.6432 : person 1 got usher
 0.6919 : person 3 has purchased the ticket 
 0.6919 : person 3 waiting for usher
 0.6919 : person 3 got usher
 0.6932 : person 1